# **Neural networks for CATE: TARNet, CFRNet and DragonNet**

Practice for the module [“Heterogeneous effects”](https://ai-interpretability.school), lesson “Neural networks for CATE”.

In the lesson the three architectures were diagrams. TARNet ([Shalit, Johansson and Sontag, 2017](https://arxiv.org/abs/1606.03976)) learns a shared representation $\Phi(x)$ and two outcome heads, one per group. CFRNet adds to the loss a penalty on the difference between the distributions of the representation in the two groups. DragonNet ([Shi, Blei and Veitch, 2019](https://arxiv.org/abs/1906.02120)) adds a third head — the propensity — and targeted regularization, a correction after which the average effect estimate satisfies the estimating equation of the doubly robust estimator. Here we build all three in `torch` and check the claims of the lesson:

1. the shared part of the network gives a better effect estimate than the two separate models of the T-learner;
2. the CFRNet penalty brings the representations of the groups closer, and affects the effect estimation error less;
3. targeted regularization reduces the error of the average effect, not of CATE;
4. one network with one seed is not a result: the spread between seeds is of the same order as the difference between architectures.

Plan of the notebook:

1. the IHDP data — the same as in the notebook “R-learner”;
2. a shared network and training with early stopping; TARNet;
3. CFRNet: a penalty on the difference of representations;
4. DragonNet and targeted regularization;
5. where it breaks: training without early stopping and the spread over seeds;
6. implementations.

The notebook runs on a CPU; the whole run takes three to six minutes: 35 small networks, each trained in a few seconds.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

SEED = 0
torch.set_num_threads(1)     # one thread: the same order of additions on every run
pd.set_option('display.width', 140)

## 1. The IHDP data

The semi-synthetic set of [Hill (2011)](https://doi.org/10.1198/jcgs.2010.08162) built on the features of the real IHDP experiment: 747 children, 139 in the treatment group, 25 features; the outcomes are generated by a known formula, so every child's true effect $\tau(x)=\mu_1(x)-\mu_0(x)$ is known. We take the first realization from the [CEVAE repository](https://github.com/AMLab-Amsterdam/CEVAE) (Louizos et al., 2017) and the same 70/30 split as in the notebook “R-learner”: there, on this held-out part, the S-learner gave a PEHE of 0.459 and the T-learner on gradient boosting gave 0.859.

We measure the error as in the notebook “R-learner”: $\mathrm{PEHE}=\sqrt{\tfrac1n\sum_i(\hat\tau(x_i)-\tau(x_i))^2}$ on the held-out part (in Shalit et al. this is $\sqrt{\epsilon_{\mathrm{PEHE}}}$), and the error of the average effect $|\bar{\hat\tau}-\bar\tau|$ on the same part. For early stopping we set aside another 20% of the training part — a validation part on which we track the error of predicting the observed outcome: the network never sees the true effect during training.

In [ ]:
URL = 'https://raw.githubusercontent.com/AMLab-Amsterdam/CEVAE/master/datasets/IHDP/csv/ihdp_npci_1.csv'
cols = ['treatment', 'y_factual', 'y_cfactual', 'mu0', 'mu1'] + [f'x{i}' for i in range(1, 26)]
df = pd.read_csv(URL, header=None, names=cols)
X = df[[f'x{i}' for i in range(1, 26)]].values
T = df.treatment.values.astype(int)
Y = df.y_factual.values
tau = (df.mu1 - df.mu0).values

tr, te = train_test_split(np.arange(len(df)), test_size=0.3, random_state=SEED, stratify=T)
fit_idx, val_idx = train_test_split(tr, test_size=0.2, random_state=SEED, stratify=T[tr])
print('training:', len(fit_idx), '| validation for early stopping:', len(val_idx), '| held-out part:', len(te))
print('share of the treatment group in training:', round(T[fit_idx].mean(), 3))

## 2. A shared network and TARNet

One network for all three architectures:

- the representation $\Phi(x)$: two layers of 100 neurons with ELU activation, followed by normalizing each row to unit length — Shalit et al. normalize $\Phi$ because otherwise the CFRNet penalty depends on the scale of the representation;
- two outcome heads $h_0(\Phi)$ and $h_1(\Phi)$: a layer of 50 neurons and an output each;
- a propensity head $g(\Phi)=\sigma(w^\top\Phi+b)$ — linear, as in Shi et al.; TARNet and CFRNet do not use it;
- a parameter $\varepsilon$ for targeted regularization.

Training uses the full batch, Adam with rate $10^{-3}$ and `weight_decay` $10^{-4}$, up to 3000 steps; we keep the weights from the step with the smallest error of predicting the observed outcome on the validation part. The outcome is standardized on the training part and rescaled back at prediction.

The TARNet loss is a weighted error on the observed outcome: head $h_{t_i}$ predicts $y_i$, and the weights $w_i=\frac{t_i}{2u}+\frac{1-t_i}{2(1-u)}$, where $u$ is the share of the treatment group, equalize the contribution of the two groups (Shalit et al., equation (3)):

$$\mathcal L_{\mathrm{TAR}}=\frac1n\sum_i w_i\,\bigl(h_{t_i}(\Phi(x_i))-y_i\bigr)^2.$$

The function `fit` below trains all three architectures; you will write the CFRNet penalty `ipm_linear` in section 3 and the effect estimate after targeted regularization in section 4.

In [ ]:
class Net(nn.Module):
    def __init__(self, d, h=100, hh=50):
        super().__init__()
        self.phi = nn.Sequential(nn.Linear(d, h), nn.ELU(), nn.Linear(h, h), nn.ELU())
        self.h0 = nn.Sequential(nn.Linear(h, hh), nn.ELU(), nn.Linear(hh, 1))
        self.h1 = nn.Sequential(nn.Linear(h, hh), nn.ELU(), nn.Linear(hh, 1))
        self.g = nn.Linear(h, 1)
        self.eps = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        z = self.phi(x)
        z = z / z.norm(dim=1, keepdim=True)
        return z, self.h0(z).squeeze(1), self.h1(z).squeeze(1), torch.sigmoid(self.g(z)).squeeze(1)


def tensors(idx, ym, ys):
    return (torch.tensor(X[idx], dtype=torch.float32), torch.tensor(T[idx], dtype=torch.float32),
            torch.tensor((Y[idx] - ym) / ys, dtype=torch.float32))


def fit(kind='tarnet', alpha=0.0, beta=0.0, seed=SEED, steps=3000, early_stop=True):
    # kind: 'tarnet' (with alpha > 0 this is CFRNet) or 'dragonnet' (with beta > 0 — targeted regularization)
    torch.manual_seed(seed)
    net = Net(X.shape[1])
    ym, ys = Y[fit_idx].mean(), Y[fit_idx].std()
    x, t, y = tensors(fit_idx, ym, ys)
    xv, tv, yv = tensors(val_idx, ym, ys)
    u = t.mean()
    w = t / (2 * u) + (1 - t) / (2 * (1 - u))
    opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-4)
    best, state = np.inf, None
    for step in range(steps):
        opt.zero_grad()
        z, q0, q1, g = net(x)
        q = t * q1 + (1 - t) * q0
        if kind == 'dragonnet':
            # Shi et al., (2.2): outcome error plus alpha * propensity cross-entropy
            loss = ((q - y) ** 2).mean() + alpha * nn.functional.binary_cross_entropy(g.clamp(1e-3, 1 - 1e-3), t)
            if beta > 0:
                # Shi et al., (3.2): Q~ = Q + eps * (t/g - (1-t)/(1-g))
                gc = g.clamp(0.01, 0.99)
                q_tilde = q + net.eps * (t / gc - (1 - t) / (1 - gc))
                loss = loss + beta * ((y - q_tilde) ** 2).mean()
        else:
            loss = (w * (q - y) ** 2).mean()
            if alpha > 0:
                loss = loss + alpha * ipm_linear(z, t)
        loss.backward()
        opt.step()
        with torch.no_grad():
            _, v0, v1, _ = net(xv)
            val = float(((tv * v1 + (1 - tv) * v0 - yv) ** 2).mean())
        if val < best or not early_stop:
            best, best_step, state = val, step, {k: v.clone() for k, v in net.state_dict().items()}
    net.load_state_dict(state)
    with torch.no_grad():
        z, q0, q1, g = net(torch.tensor(X, dtype=torch.float32))
    # predictions for all 747 children, on the original scale
    return {'q0': q0.numpy() * ys + ym, 'q1': q1.numpy() * ys + ym, 'g': g.numpy(), 'eps': float(net.eps.detach()) * ys,
            'z': z.numpy(), 'best_step': best_step}


SEEDS = range(5)

**Task 1. Write a function `errors(cate)` that returns the PEHE and the error of the average effect on the held-out part `te`. Train TARNet (`fit('tarnet')`) with seeds 0–4; the effect estimate is `q1 - q0`. Print the PEHE and the error of the average effect for every seed and their means. What is the mean PEHE of TARNet over the five seeds?**

In [ ]:
# Your code here

**Your answer:**

## 3. CFRNet: a penalty on the difference of representations

Theorem 1 of Shalit et al. bounds $\epsilon_{\mathrm{PEHE}}$ from above by the sum of the errors of predicting the observed outcome in the two groups and the distance between the distributions of the representation $\Phi$ among the treated and the controls — an integral probability metric (IPM). CFRNet adds this distance to the loss with weight $\alpha$:

$$\mathcal L_{\mathrm{CFR}}=\mathcal L_{\mathrm{TAR}}+\alpha\cdot\mathrm{IPM}\bigl(\{\Phi(x_i)\}_{t_i=0},\{\Phi(x_i)\}_{t_i=1}\bigr).$$

The authors use the Wasserstein distance or MMD. The simplest option is the squared linear MMD, which is the squared distance between the mean representations of the groups (“CFR MMD”, “squared linear MMD” in Table 1 of the paper):

$$\mathrm{MMD}^2_{\mathrm{lin}}=\Bigl\|\frac1{n_1}\sum_{t_i=1}\Phi(x_i)-\frac1{n_0}\sum_{t_i=0}\Phi(x_i)\Bigr\|^2.$$

With $\alpha=0$ this is TARNet.

**Task 2. Write `ipm_linear(z, t)` in `torch`. Train CFRNet with $\alpha\in\{1, 10, 100\}$ and seeds 0–4. For TARNet and each $\alpha$ print the means over seeds of the PEHE, the error of the average effect, and the linear MMD$^2$ of the representation on the training part `fit_idx`. How does a growing $\alpha$ change the distance between representations, and how does it change the PEHE?**

In [ ]:
# Your code here

**Your answer:**

## 4. DragonNet and targeted regularization

DragonNet trains the propensity head together with the outcome heads (Shi et al., equation (2.2)):

$$\hat R(\theta)=\frac1n\sum_i\bigl(Q(t_i,x_i)-y_i\bigr)^2+\alpha\cdot\mathrm{CrossEntropy}\bigl(g(x_i),t_i\bigr),$$

where $Q(t,x)=h_t(\Phi(x))$. The authors' argument: adjusting for confounders only needs the propensity (the theorem of Rosenbaum and Rubin), so it helps to compress the representation to the information that predicts the treatment. The average effect estimate is $\hat\psi_Q=\frac1n\sum_i\bigl(Q(1,x_i)-Q(0,x_i)\bigr)$.

Targeted regularization adds one parameter $\varepsilon$ and a term with weight $\beta$ (equations (3.2)–(3.4)):

$$\tilde Q(t,x)=Q(t,x)+\varepsilon\Bigl(\frac{t}{g(x)}-\frac{1-t}{1-g(x)}\Bigr),\qquad \hat R(\theta)+\beta\,\frac1n\sum_i\bigl(y_i-\tilde Q(t_i,x_i)\bigr)^2.$$

The derivative with respect to $\varepsilon$ is zero at the minimum, and that is exactly the estimating equation of the doubly robust average effect estimator with the models $\tilde Q$ and $g$. The object's effect estimate after the correction is $\tilde Q(1,x)-\tilde Q(0,x)=Q(1,x)-Q(0,x)+\varepsilon\bigl(\frac1{g(x)}+\frac1{1-g(x)}\bigr)$. Shi et al. use $\alpha=\beta=1$.

**Task 3. Train DragonNet (`fit('dragonnet', alpha=1)`) and DragonNet with targeted regularization (`fit('dragonnet', alpha=1, beta=1)`) with seeds 0–4. Write `cate_treg(r)` — the effect estimate from $\tilde Q$ (clip the propensity to $[0.01, 0.99]$, as in training). Collect in one table the means over seeds of the PEHE and the error of the average effect for TARNet, CFRNet with $\alpha=10$, DragonNet, and DragonNet with targeted regularization; for the last one print $\hat\varepsilon$ of every seed as well. What is the mean error of the average effect for DragonNet with targeted regularization?**

In [ ]:
# Your code here

**Your answer:**

## 5. Where it breaks

### 5.1. Without early stopping

On 418 training objects a network with two layers of 100 neurons easily memorizes the noise in the observed outcomes. The error of predicting the outcome is visible on the validation part, but the error of the effect is not: it can only be computed on semi-synthetic data.

### 5.2. The spread over seeds

All the comparisons above are means over five seeds. The seed changes the initial weights, and with them the minimum that is found.

**Task 4. Train TARNet without early stopping (`fit('tarnet', early_stop=False)`, all 3000 steps) with seeds 0–4 and print the mean PEHE. Then, for TARNet, CFRNet with $\alpha=10$ and DragonNet, print the PEHE of each of the five seeds and the spread (maximum minus minimum). What do these numbers say about comparing architectures by one seed?**

In [ ]:
# Your code here

**Your answer:**

## 6. Implementations

- [CATENets](https://github.com/AliciaCurth/CATENets) (BSD-3-Clause licence, Curth and van der Schaar): `catenets.models.torch.TARNet`, `DragonNet`, `FlexTENet` and neural metalearners in `torch`, plus JAX versions.
- [`causalml`](https://github.com/uber/causalml): `causalml.inference.tf.DragonNet` in TensorFlow and `causalml.inference.torch.CEVAE`.
- The authors' code: [cfrnet](https://github.com/clinicalml/cfrnet) (MIT licence, TensorFlow 1) and [dragonnet](https://github.com/claudiashi57/dragonnet) — the DragonNet repository has no licence, so the notebook uses its own implementation.

## What to take away

- **A shared representation and two outcome heads** — TARNet — already beat the T-learner on boosting on these data, but not the S-learner.
- **The CFRNet penalty brings the group representations closer,** and on these data the PEHE and the error of the average effect fall together with the distance; but the gain in PEHE is smaller than the spread between seeds, and on real data $\alpha$ has to be chosen without the true effect.
- **Targeted regularization works on the average effect,** not on the effect of a single object — it solves the estimating equation for the average.
- **A neural CATE estimator cannot be judged by one seed or without early stopping:** the spread between seeds is of the same order as the difference between architectures, and overfitting shows only in the effect error, which real data do not have.

Sources:
1. [Johansson, Shalit & Sontag, 2016. Learning representations for counterfactual inference](https://arxiv.org/abs/1605.03661)
2. [Shalit, Johansson & Sontag, 2017. Estimating individual treatment effect: generalization bounds and algorithms](https://arxiv.org/abs/1606.03976)
3. [Shi, Blei & Veitch, 2019. Adapting neural networks for the estimation of treatment effects](https://arxiv.org/abs/1906.02120)
4. [Hill, 2011. Bayesian nonparametric modeling for causal inference](https://doi.org/10.1198/jcgs.2010.08162)
5. [Louizos, Shalit, Mooij & Sontag, 2017. Causal effect inference with deep latent-variable models](https://arxiv.org/abs/1705.08821); the IHDP data — the first of ten replications in the [CEVAE repository](https://github.com/AMLab-Amsterdam/CEVAE)